In [ ]:
from sklearn.pipeline import make_pipeline
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.impute import KNNImputer
import seaborn as sns

In [ ]:
sns.set_theme(style='ticks')

In [ ]:
class InitialProcessing:
    def __init__(self, file_path):
        self.file_path = file_path
        self.df = None
        self.df_cleaned = None

    def load_and_transpose(self):
        self.df = pd.read_csv(self.file_path)
        self.df = self.df.transpose()
        new_header = self.df.iloc[0]
        self.df = self.df[1:]
        self.df.columns = new_header
        print(f"Original number of features: {len(self.df.columns)} features")

    def clean_data(self):
        self.df_cleaned = self.df.dropna(axis = 1, how = 'all')
        print(f"{self.__class__.__name__} keeping {len(self.df_cleaned.columns)} features")

    def patient_id(self):
        new_index = self.df_cleaned.index.str.split('-').str[:3].str.join('-')
        self.df_cleaned.index = new_index

    def process_data(self):
        self.load_and_transpose()
        self.clean_data()
        self.patient_id()
        return self.df_cleaned


class RemoveFeaturesWithZeros(BaseEstimator, TransformerMixin):

    def __init__(self, threshold: float, verbose: bool = False):
        self.threshold = threshold
        self.verbose = verbose

    def fit(self, X, y = None):
        self.columns_ = X.columns[(X == 0).sum(axis=0) / len(X) < self.threshold]
        if self.verbose:
            print(f"{self.__class__.__name__} keeping {len(self.columns_)} features")
        return self

    def transform(self, X, y = None):
        transformed_X = X[self.columns_]
        return transformed_X


class RemoveFeaturesWithNaN(BaseEstimator, TransformerMixin):

    def __init__(self, threshold: float, verbose: bool = False):
        self.threshold = threshold
        self.verbose = verbose

    def fit(self, X, y = None):
        self.columns_ = X.columns[(X.isna()).sum(axis=0) / len(X) < self.threshold]
        if self.verbose:
            print(f"{self.__class__.__name__} keeping {len(self.columns_)} features")
        return self

    def transform(self, X, y = None):
        transformed_X = X[self.columns_]
        return transformed_X


class RemoveFeaturesLowMAE(BaseEstimator, TransformerMixin):

    def __init__(self, percentage_to_keep: float, verbose: bool = False):
        self.percentage_to_keep = percentage_to_keep
        self.verbose = verbose

    def fit(self, X, y = None):
        X = X.apply(pd.to_numeric, errors='coerce')
        var = np.abs(X - np.median(X, axis = 0))
        var = np.median(var, axis = 0)
        var = pd.Series(var, index = X.columns)
        columns = var.nlargest(n = int(X.shape[1] * self.percentage_to_keep)).index
        self.columns_ = X.columns.intersection(columns)
        if self.verbose:
            print(f"{self.__class__.__name__} keeping {len(self.columns_)} features")
        return self

    def transform(self, X, y = None):
        transformed_X = X[self.columns_]
        return transformed_X


class RemoveCorrelatedFeatures(BaseEstimator, TransformerMixin):

    def __init__(self, threshold: float, verbose: bool = False):
        self.threshold = threshold
        self.verbose = verbose

    def fit(self, X, y = None):
        X = X.apply(pd.to_numeric, errors='coerce')
        corr_mat = np.abs(np.corrcoef(X.T.values))
        np.fill_diagonal(corr_mat, 0)
        upper = pd.DataFrame(corr_mat, index = X.columns, columns = X.columns)
        self.columns_to_drop_ = upper.columns[(upper > self.threshold).any()]
        if self.verbose:
            print(f"{self.__class__.__name__} keeping {len(X.columns) - len(self.columns_to_drop_)} features")
        return self

    def transform(self, X, y=None):
        transformed_X = X.drop(columns = self.columns_to_drop_)
        return transformed_X


class Log2Transformation(FunctionTransformer):

    def __init__(self):
        super().__init__(lambda x: np.log2(1 + x.astype(float)))


class GeneMutations(BaseEstimator, TransformerMixin):
    def __init__(self, verbose: bool = False):
        self.verbose = verbose
        self.sorted_genes_ = []

    def fit(self, X, y = None):
        stacked_genes = X.stack()
        genes = stacked_genes.unique()
        gene_counts = stacked_genes.value_counts()
        self.sorted_genes_ = gene_counts.sort_values(ascending=False).index.tolist()
        if self.verbose:
            print(f"{self.__class__.__name__} has {len(self.sorted_genes_)} genes")
        return self

    def transform(self, X, y = None):
        transformed_X = pd.DataFrame(0, index = X.index, columns = self.sorted_genes_)
        for idx, row in X.iterrows():
            for gene in row.dropna():
                if gene in transformed_X.columns:
                    transformed_X.at[idx, gene] = 1
        return transformed_X
        

class ValueImputation(BaseEstimator, TransformerMixin):
    def __init__(self, verbose: bool = False):
        self.verbose = verbose
        self.imputer = KNNImputer()
        self.scaler = StandardScaler()

    def fit(self, X, y = None):
        X_scaled = self.scaler.fit_transform(X)
        self.imputer.fit(X_scaled)
        return self

    def transform(self, X, y = None):
        missing_values = X.isna().sum().sum()
        X_scaled = self.scaler.transform(X)
        X_imputed = self.imputer.transform(X_scaled)
        X_inverse = self.scaler.inverse_transform(X_imputed)
        transformed_X = pd.DataFrame(X_inverse, columns = X.columns, index = X.index)
        if self.verbose:
            print(f"{self.__class__.__name__} transforming {missing_values} values")
        return transformed_X

In [ ]:
cna_data = InitialProcessing("raw data/cancer_data_PAAD_GISTIC_Peaks-20160128.csv").process_data()
cna_data.columns = cna_data.loc['Descriptor']
for col in cna_data.columns:
    if cna_data.loc['type', col] == 'Deletion':
        cna_data[col] = cna_data[col].apply(lambda x: -x if isinstance(x, (int, float)) and x != 0 else x)
cna_new = cna_data[cna_data.index.str.contains('TCGA')]
cna_new = cna_new.apply(pd.to_numeric)
cna_new

In [ ]:
RPPA_data = InitialProcessing("raw data/cancer_data_PAAD_RPPAArray-20160128.csv").process_data()
RPPA_pipeline = make_pipeline(
    RemoveFeaturesWithNaN(threshold=0.2, verbose=True),
    ValueImputation(verbose=True)
)
RPPA_new = RPPA_pipeline.fit_transform(RPPA_data)
RPPA_new

In [ ]:
miRNA_data = InitialProcessing("raw data/cancer_data_PAAD_miRNASeqGene-20160128.csv").process_data()
miRNA_pipeline = make_pipeline(
    RemoveFeaturesWithZeros(threshold=0.2, verbose=True),
    Log2Transformation()
)
miRNA_new = miRNA_pipeline.fit_transform(miRNA_data)
miRNA_new

In [ ]:
RNAseq_data = InitialProcessing("raw data/cancer_data_PAAD_RNASeq2GeneNorm-20160128.csv").process_data()
RNAseq_pipeline = make_pipeline(
    RemoveFeaturesWithZeros(threshold=0.2, verbose=True),
    RemoveFeaturesLowMAE(percentage_to_keep=0.1, verbose=True),
    RemoveCorrelatedFeatures(threshold=0.85, verbose=True),
    Log2Transformation()
)
RNAseq_new = RNAseq_pipeline.fit_transform(RNAseq_data)
RNAseq_new

In [ ]:
methylation_data = InitialProcessing("raw data/cancer_data_PAAD_Methylation-20160128.csv").process_data()
methylation_pipeline = make_pipeline(
    RemoveFeaturesWithNaN(threshold=0.2, verbose=True),
    RemoveFeaturesLowMAE(percentage_to_keep=0.01, verbose=True),
    RemoveCorrelatedFeatures(threshold=0.85, verbose=True)
)
methylation_new = methylation_pipeline.fit_transform(methylation_data)
methylation_new

In [ ]:
mutations_data = InitialProcessing("raw data/cancer_data_PAAD_Mutation-20160128.csv").process_data()
mutations_pipeline = make_pipeline(
    GeneMutations(verbose=True),
    RemoveFeaturesWithZeros(threshold=0.95, verbose=True)
)
mutations_new = mutations_pipeline.fit_transform(mutations_data)
mutations_new

### Creating UpSet plot

In [ ]:
upset_df = pd.DataFrame(columns=['CNA', 'miRNA', 'Mutations', 'Methylation', 'RNA-seq', 'RPPA'])
upset_df['Patient ID'] = pd.merge(cna_new, mutations_new, how='outer', left_index=True, right_index=True).index
upset_df['CNA'] = upset_df['Patient ID'].isin(cna_new.index)
upset_df['miRNA'] = upset_df['Patient ID'].isin(miRNA_new.index)
upset_df['Mutations'] = upset_df['Patient ID'].isin(mutations_new.index)
upset_df['Methylation'] = upset_df['Patient ID'].isin(methylation_new.index)
upset_df['RNA-seq'] = upset_df['Patient ID'].isin(RNAseq_new.index)
upset_df['RPPA'] = upset_df['Patient ID'].isin(RPPA_new.index)
upset_df.set_index(['CNA', 'miRNA', 'Mutations', 'Methylation', 'RNA-seq', 'RPPA'], inplace=True, drop=True)
upset_df

In [ ]:
import upsetplot
from upsetplot import plot
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(9, 5))
plot(upset_df, fig=fig, element_size=None, sort_by='cardinality', show_counts=True)
for ax in fig.axes:
    ax.grid(False)
plt.savefig('figures/upsetplot.svg', bbox_inches='tight')
plt.show()

### Survival and disease-free plots for all patients (no clusters)

In [ ]:
from lifelines import KaplanMeierFitter
import matplotlib.pyplot as plt
from lifelines.plotting import add_at_risk_counts

In [ ]:
final_benchmarking = pd.merge(methylation_new, cna_new, how='outer', left_index=True, right_index=True)
patients_list = final_benchmarking.index.tolist()

In [ ]:
colorblind_palette = sns.color_palette("colorblind")
clinical_data = pd.read_csv('raw data/cancer_data_PAAD_clinical_data.tsv', sep='\t')
survival_data = clinical_data[['Patient ID', 'Overall Survival Status', 'Overall Survival (Months)']]
survival_data.set_index('Patient ID', inplace=True)
survival_data['Overall Survival Status'] = survival_data['Overall Survival Status'].str.split(':').str[0].astype(int)
survival_data = survival_data.loc[survival_data.index.isin(patients_list)]

kmf = KaplanMeierFitter()
kmf.fit(survival_data['Overall Survival (Months)'], event_observed=survival_data['Overall Survival Status'])
plt.figure(figsize=(10, 5))
ax = kmf.plot_survival_function(show_censors=True, legend=False, ci_show=False, color=colorblind_palette[0])
max_time = max(survival_data['Overall Survival (Months)'])
ticks = np.arange(0, max_time + 6, 6)
ax.set_xticks(ticks)
add_at_risk_counts(kmf, ax=plt.gca())
ax.set_ylabel('Survival probability')
ax.set_xlabel('Time (months)')
plt.tight_layout()
plt.savefig('figures/survival_all.svg', bbox_inches='tight')
plt.show()

In [ ]:
diseasefree_data = clinical_data[['Patient ID', 'Disease Free Status', 'Disease Free (Months)']]
diseasefree_data.set_index('Patient ID', inplace=True)
diseasefree_data.dropna(how='any', axis=0, inplace=True)
diseasefree_data['Disease Free Status'] = diseasefree_data['Disease Free Status'].str.split(':').str[0].astype(int)
diseasefree_data = diseasefree_data.loc[diseasefree_data.index.isin(patients_list)]

kmf = KaplanMeierFitter()
kmf.fit(diseasefree_data['Disease Free (Months)'], event_observed=diseasefree_data['Disease Free Status'])
plt.figure(figsize=(10, 5))
ax = kmf.plot_survival_function(show_censors=True, legend=False, ci_show=False, color=colorblind_palette[0])
max_time = max(diseasefree_data['Disease Free (Months)'])
ticks = np.arange(0, max_time + 3, 6)
ax.set_xticks(ticks)
add_at_risk_counts(kmf, ax=plt.gca())
ax.set_ylabel('Disease free probability')
ax.set_xlabel('Time (months)')
plt.tight_layout()
plt.savefig('figures/diseasefree_all.svg', bbox_inches='tight')
plt.show()